# 예제 franka_ex09: FR3 충돌 객체 (Collision Objects)

`PlanningScene` 에 박스/원기둥 장애물을 넣고, FR3 가 그 사이를 빠져나가는 경로를 찾도록 한다.
6-DOF 용 `ex09_collision_objects.py` 를 FR3 워크스페이스에 맞게 옮겨왔다.

**6-DOF 예제와 다른 점**
- 장애물 크기/위치를 FR3 reach 에 맞게 키움 — 벽: x=0.40, 높이 25cm; 기둥: x=0.40, 높이 60cm
- 끝단 링크 `fr3_hand_tcp`, frame `fr3_link0`
- `home` 없음 → 시작/복귀는 `ready`

**학습 내용**
- `ApplyPlanningScene` 서비스로 장애물 추가/제거
- `CollisionObject.ADD` / `REMOVE`
- `SolidPrimitive(BOX, CYLINDER)`
- 장애물 회피 경로 — RViz Planning Scene Display 에서 충돌 객체와 trajectory 동시 관찰

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/collision_demo_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

충돌 객체 자체는 RViz **Planning Scene Display** 에서 보인다 (이 노트북이 마커로 따로 그리지 않음).

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/collision_demo_markers'

## 2. 초기화

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from moveit_msgs.srv import ApplyPlanningScene
from moveit_msgs.msg import (
    PlanningScene, CollisionObject,
)
from shape_msgs.msg import SolidPrimitive
from visualization_msgs.msg import MarkerArray
from geometry_msgs.msg import Point, Quaternion

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex09_collision_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex09 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)
scene_client = node.create_client(ApplyPlanningScene, 'apply_planning_scene')

## 3. 서버 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    if not scene_client.wait_for_service(timeout_sec=timeout_sec):
        raise RuntimeError('apply_planning_scene 서비스 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + planning_scene + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

## 6. CollisionObject 헬퍼

`ApplyPlanningScene` 서비스에 `PlanningScene(is_diff=True)` 로 보내면
기존 Scene 위에 누적/제거 된다 (`is_diff=False` 면 통째 교체).

In [ ]:
def make_box(object_id: str, position, dimensions,
             frame_id: str = REFERENCE_FRAME) -> CollisionObject:
    co = CollisionObject()
    co.header.frame_id = frame_id
    co.id = object_id
    co.operation = CollisionObject.ADD
    box = SolidPrimitive()
    box.type = SolidPrimitive.BOX
    box.dimensions = list(dimensions)
    co.primitives.append(box)
    pose = Pose()
    pose.position = Point(x=position[0], y=position[1], z=position[2])
    pose.orientation = Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
    co.primitive_poses.append(pose)
    return co

def make_cylinder(object_id: str, position, height: float, radius: float,
                  frame_id: str = REFERENCE_FRAME) -> CollisionObject:
    co = CollisionObject()
    co.header.frame_id = frame_id
    co.id = object_id
    co.operation = CollisionObject.ADD
    cyl = SolidPrimitive()
    cyl.type = SolidPrimitive.CYLINDER
    cyl.dimensions = [height, radius]
    co.primitives.append(cyl)
    pose = Pose()
    pose.position = Point(x=position[0], y=position[1], z=position[2])
    pose.orientation = Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)
    co.primitive_poses.append(pose)
    return co

def apply_diff(world_objects=None) -> bool:
    scene = PlanningScene()
    scene.is_diff = True
    if world_objects:
        scene.world.collision_objects = list(world_objects)
    req = ApplyPlanningScene.Request()
    req.scene = scene
    fut = scene_client.call_async(req)
    rclpy.spin_until_future_complete(node, fut)
    return fut.result().success

def add_object(co: CollisionObject) -> bool:
    return apply_diff([co])

def remove_object(object_id: str, frame_id: str = REFERENCE_FRAME) -> bool:
    co = CollisionObject()
    co.header.frame_id = frame_id
    co.id = object_id
    co.operation = CollisionObject.REMOVE
    return apply_diff([co])

def clear_all() -> bool:
    co = CollisionObject()
    co.header.frame_id = REFERENCE_FRAME
    co.id = ''
    co.operation = CollisionObject.REMOVE
    return apply_diff([co])

## 7. ready 자세로 초기화

In [ ]:
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target)
time.sleep(1.0)

## 8. 시나리오 1 — 낮은 벽 넘기

base 로부터 40cm 앞에 폭 40cm × 높이 25cm 벽을 두고,
벽 너머의 위치(`x=0.55`)로 가는 경로를 계획한다.
플래너는 자연스럽게 벽 위로 호를 그리며 도달한다.

In [ ]:
wall = make_box('wall', position=(0.40, 0.0, 0.13),
                dimensions=(0.04, 0.40, 0.25))
add_object(wall)
node.get_logger().info('  벽 추가 (x=0.40, 높이 25cm, 폭 40cm)')
time.sleep(1.0)

target1 = make_pose(0.55, 0.0, 0.40, math.pi, 0.0, 0.0)
node.get_logger().info('--- 벽 너머 목표로 이동 ---')
ok = go_to_pose_goal(target1)
node.get_logger().info(f'  결과: {"성공 (벽 위 회피)" if ok else "실패"}')
time.sleep(1.0)

### 8-1. ready 복귀 + 벽 제거

In [ ]:
go_to_joint_goal(ready_target)
time.sleep(0.5)
remove_object('wall')
node.get_logger().info('  벽 제거됨')
time.sleep(0.5)

## 9. 시나리오 2 — 옆 기둥 우회

좌측에 기둥을 두고, 끝단을 좌측 깊숙이 보내려 하면 플래너가 기둥을 우회한다.

In [ ]:
pillar = make_box('pillar', position=(0.40, 0.10, 0.30),
                  dimensions=(0.06, 0.06, 0.60))
add_object(pillar)
node.get_logger().info('  기둥 추가 (x=0.40, y=+0.10, 높이 60cm)')
time.sleep(1.0)

target2 = make_pose(0.40, 0.25, 0.40, math.pi, 0.0, 0.0)
node.get_logger().info('--- 기둥 너머 좌측 목표로 이동 ---')
ok = go_to_pose_goal(target2)
node.get_logger().info(f'  결과: {"성공 (우회 경로)" if ok else "실패"}')
time.sleep(1.0)

go_to_joint_goal(ready_target)
time.sleep(0.5)
remove_object('pillar')
node.get_logger().info('  기둥 제거됨')
time.sleep(0.5)

## 10. 시나리오 3 — 원기둥 회피

가는 원기둥(반경 5cm) 을 정면에 놓고 그 옆으로 끝단을 보낸다.

In [ ]:
cylinder = make_cylinder('cyl', position=(0.45, 0.0, 0.30),
                          height=0.50, radius=0.05)
add_object(cylinder)
node.get_logger().info('  원기둥 추가 (x=0.45, r=5cm, h=50cm)')
time.sleep(1.0)

target3 = make_pose(0.45, -0.20, 0.40, math.pi, 0.0, 0.0)
node.get_logger().info('--- 원기둥 우측 목표로 이동 ---')
ok = go_to_pose_goal(target3)
node.get_logger().info(f'  결과: {"성공 (회피)" if ok else "실패"}')
time.sleep(1.0)

## 11. 정리 — 모든 객체 제거 + ready 복귀

In [ ]:
go_to_joint_goal(ready_target)
clear_all()
node.get_logger().info('=== franka_ex09 완료! ===')

## 12. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass